=============================================================================
Toronto Shelter System — Fragility Index & Scenario Simulation

Q4: System Fragility Index — which sectors are most operationally at risk?
Q5: Scenario Simulation — what happens if demand increases or capacity drops?
=============================================================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 130

=============================================================================
LOAD DATA
=============================================================================

In [ ]:
print("Loading data...")
df = pd.read_excel('public_services_dataset.xlsx')
df['OCCUPANCY_DATE'] = pd.to_datetime(df['OCCUPANCY_DATE'])
df['EFFECTIVE_CAPACITY'] = (df['ACTUAL_CAPACITY'] - df['UNAVAILABLE_CAPACITY']).clip(lower=1)
df['EFFECTIVE_OCCUPANCY_RATE'] = (df['OCCUPIED_CAPACITY'] / df['EFFECTIVE_CAPACITY']).clip(upper=1.5)
df['AT_FULL_CAPACITY'] = df['OCCUPANCY_RATE'] >= 1.0
df['AVAILABLE_BUFFER'] = df['ACTUAL_CAPACITY'] - df['OCCUPIED_CAPACITY']  # free beds

In [ ]:
print(f"  Rows loaded: {len(df):,}\n")

=============================================================================
Q4: SYSTEM FRAGILITY INDEX
=============================================================================
The index combines three components per sector:
  Component 1 — Average occupancy rate         (higher = worse)
  Component 2 — % of days at full capacity     (higher = worse)
  Component 3 — Average unavailable capacity   (higher = worse)

Each component is normalized to 0–1 using min-max scaling,
then averaged into a single Fragility Score (0 = safest, 1 = most fragile).
=============================================================================

In [ ]:
print("=" * 60)
print("Q4: System Fragility Index")
print("=" * 60)

In [ ]:
sector_stats = (
    df.groupby('SECTOR')
    .agg(
        Avg_Occupancy_Rate   = ('OCCUPANCY_RATE',         'mean'),
        Pct_Days_Full        = ('AT_FULL_CAPACITY',       'mean'),
        Avg_Unavailable      = ('UNAVAILABLE_CAPACITY',   'mean'),
        Avg_Buffer           = ('AVAILABLE_BUFFER',       'mean'),
        Avg_Effective_Rate   = ('EFFECTIVE_OCCUPANCY_RATE','mean'),
    )
    .reset_index()
)

In [ ]:
def minmax(series):
    """Normalize a series to 0–1. If all values are equal, return 0.5."""
    rng = series.max() - series.min()
    if rng == 0:
        return pd.Series([0.5] * len(series), index=series.index)
    return (series - series.min()) / rng

In [ ]:
# Normalize each component (all three: higher raw value = more fragile)
sector_stats['Score_OccupancyRate'] = minmax(sector_stats['Avg_Occupancy_Rate'])
sector_stats['Score_PctDaysFull']   = minmax(sector_stats['Pct_Days_Full'])
sector_stats['Score_Unavailable']   = minmax(sector_stats['Avg_Unavailable'])

In [ ]:
# Weighted average — occupancy rate and pct days full are more operationally
# meaningful than unavailable beds, so we weight them more
WEIGHTS = {
    'Score_OccupancyRate': 0.40,
    'Score_PctDaysFull':   0.40,
    'Score_Unavailable':   0.20,
}

In [ ]:
sector_stats['Fragility_Score'] = (
    sector_stats['Score_OccupancyRate'] * WEIGHTS['Score_OccupancyRate'] +
    sector_stats['Score_PctDaysFull']   * WEIGHTS['Score_PctDaysFull']   +
    sector_stats['Score_Unavailable']   * WEIGHTS['Score_Unavailable']
)

In [ ]:
sector_stats = sector_stats.sort_values('Fragility_Score', ascending=False).reset_index(drop=True)
sector_stats['Rank'] = sector_stats.index + 1

In [ ]:
# Risk label
def risk_label(score):
    if score >= 0.70: return 'Critical'
    if score >= 0.45: return 'High'
    if score >= 0.20: return 'Moderate'
    return 'Low'

In [ ]:
sector_stats['Risk_Level'] = sector_stats['Fragility_Score'].apply(risk_label)

In [ ]:
print(sector_stats[[
    'Rank','SECTOR','Avg_Occupancy_Rate','Pct_Days_Full',
    'Avg_Unavailable','Fragility_Score','Risk_Level'
]].round(4).to_string(index=False))
print()

In [ ]:
# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Q4: System Fragility Index by Sector', fontsize=14, fontweight='bold', y=1.01)

In [ ]:
risk_colors = {'Critical': '#d32f2f', 'High': '#f57c00', 'Moderate': '#fbc02d', 'Low': '#388e3c'}
bar_colors  = [risk_colors[r] for r in sector_stats['Risk_Level']]

In [ ]:
# Chart A: Fragility Score bar chart
ax = axes[0]
bars = ax.barh(sector_stats['SECTOR'], sector_stats['Fragility_Score'],
               color=bar_colors, edgecolor='white')
ax.set_xlabel('Fragility Score (0 = Safest, 1 = Most Fragile)')
ax.set_title('Overall Fragility Score by Sector')
ax.set_xlim(0, 1.1)
for bar, score, risk in zip(bars, sector_stats['Fragility_Score'], sector_stats['Risk_Level']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}  [{risk}]', va='center', fontsize=9)
legend_patches = [mpatches.Patch(color=c, label=l) for l, c in risk_colors.items()]
ax.legend(handles=legend_patches, title='Risk Level', loc='lower right')

In [ ]:
# Chart B: Stacked component breakdown
ax = axes[1]
components = ['Score_OccupancyRate', 'Score_PctDaysFull', 'Score_Unavailable']
labels     = ['Occupancy Rate (40%)', '% Days Full (40%)', 'Unavailable Beds (20%)']
comp_colors = ['#ef5350', '#ff8a65', '#ffd54f']
bottoms = np.zeros(len(sector_stats))
for comp, label, color in zip(components, labels, comp_colors):
    weighted_vals = sector_stats[comp] * WEIGHTS[comp]
    ax.barh(sector_stats['SECTOR'], weighted_vals, left=bottoms,
            label=label, color=color, edgecolor='white')
    bottoms += weighted_vals.values
ax.set_xlabel('Weighted Component Score')
ax.set_title('Fragility Score Breakdown by Component')
ax.legend(loc='lower right', fontsize=8)
ax.set_xlim(0, 1.0)

In [ ]:
plt.tight_layout()
plt.savefig('q4_fragility_index.png', bbox_inches='tight')
#plt.show()
plt.savefig("fragility_simulation.png", dpi=300, bbox_inches="tight")
plt.close()
print("  Saved: q4_fragility_index.png\n")

=============================================================================
Q5: SCENARIO SIMULATION
=============================================================================
Two simulations:
  Sim A — Demand increase: what if occupancy goes up by X%?
           → How many sector-days breach capacity?
  Sim B — Capacity increase: if we add Y beds to each sector,
           how does the average occupancy rate change?
=============================================================================

In [ ]:
print("=" * 60)
print("Q5: Scenario Simulation")
print("=" * 60)

In [ ]:
# --- Simulation A: Demand Increase ---
# For each scenario, scale up OCCUPIED_CAPACITY and recompute breach rate
demand_scenarios = [0, 2, 5, 8, 10]   # % demand increase

In [ ]:
print("Sim A — Demand Increase: % of records that breach capacity")
print(f"{'Sector':<15}", end='')
for s in demand_scenarios:
    print(f"  +{s}%", end='')
print()

In [ ]:
sim_a_results = {}
for sector in df['SECTOR'].unique():
    sub = df[df['SECTOR'] == sector].copy()
    row = [sector]
    rates = []
    for pct in demand_scenarios:
        simulated_occupied = sub['OCCUPIED_CAPACITY'] * (1 + pct / 100)
        breach_rate = (simulated_occupied > sub['ACTUAL_CAPACITY']).mean() * 100
        rates.append(breach_rate)
    sim_a_results[sector] = rates
    print(f"{sector:<15}", end='')
    for r in rates:
        print(f"  {r:>4.1f}%", end='')
    print()

In [ ]:
print()

In [ ]:
# --- Simulation B: Capacity Increase ---
# Add X beds to a sector and see new average occupancy rate
capacity_add_scenarios = [0, 10, 25, 50, 100]   # beds added per program on average

In [ ]:
print("Sim B — Capacity Increase: avg occupancy rate after adding beds")
print(f"{'Sector':<15}", end='')
for s in capacity_add_scenarios:
    print(f"  +{s} beds", end='')
print()

In [ ]:
sim_b_results = {}
for sector in df['SECTOR'].unique():
    sub = df[df['SECTOR'] == sector].copy()
    rates = []
    for add in capacity_add_scenarios:
        new_capacity = sub['ACTUAL_CAPACITY'] + add
        new_rate = (sub['OCCUPIED_CAPACITY'] / new_capacity).clip(upper=1.0).mean() * 100
        rates.append(new_rate)
    sim_b_results[sector] = rates
    print(f"{sector:<15}", end='')
    for r in rates:
        print(f"  {r:>6.1f}%", end='')
    print()

In [ ]:
print()

In [ ]:
# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Q5: Scenario Simulations', fontsize=14, fontweight='bold', y=1.01)

In [ ]:
palette = sns.color_palette("Set2", len(df['SECTOR'].unique()))
sectors_sorted = sector_stats['SECTOR'].tolist()   # keep consistent order

In [ ]:
# Chart A: Demand increase → breach rate
ax = axes[0]
for i, sector in enumerate(sectors_sorted):
    ax.plot(demand_scenarios, sim_a_results[sector],
            marker='o', label=sector, color=palette[i], linewidth=2)
ax.set_xlabel('Demand Increase (%)')
ax.set_ylabel('% of Records Breaching Capacity')
ax.set_title('Sim A: If Demand Increases,\nHow Often Does the System Breach Capacity?')
ax.legend(fontsize=8)
ax.set_xticks(demand_scenarios)
ax.set_xticklabels([f'+{s}%' for s in demand_scenarios])

In [ ]:
# Chart B: Capacity increase → new avg occupancy rate
ax = axes[1]
for i, sector in enumerate(sectors_sorted):
    ax.plot(capacity_add_scenarios, sim_b_results[sector],
            marker='o', label=sector, color=palette[i], linewidth=2)
ax.axhline(90, color='orange', linestyle='--', linewidth=1.2, label='90% target')
ax.axhline(95, color='red',    linestyle='--', linewidth=1.2, label='95% threshold')
ax.set_xlabel('Extra Beds Added per Program')
ax.set_ylabel('Avg Occupancy Rate (%)')
ax.set_title('Sim B: How Many Extra Beds Are Needed\nto Bring Occupancy Below 95%?')
ax.legend(fontsize=8)
ax.set_xticks(capacity_add_scenarios)
ax.set_xticklabels([f'+{s}' for s in capacity_add_scenarios])

In [ ]:
plt.tight_layout()
plt.savefig('q5_simulation.png', bbox_inches='tight')
#plt.show()
plt.savefig("plot.png", dpi=300, bbox_inches="tight")
plt.close()
print("  Saved: q5_simulation.png\n")

In [ ]:
# --- Summary insight ---
print("Key Insight — Beds needed to bring each sector below 95% occupancy:")
for sector in sectors_sorted:
    rates = sim_b_results[sector]
    below_95 = next((capacity_add_scenarios[i] for i, r in enumerate(rates) if r < 95), '>100')
    print(f"  {sector:<15}: +{below_95} beds per program")

In [ ]:
print("\nDone! Files saved: q4_fragility_index.png, q5_simulation.png")